# SkylineGeolocation — Full Pipeline on Colab

Stages (each skips if output already exists in Drive):
1. Setup — install HORAYZON + Python deps
2. DEM download — needs Earthdata credentials
3. Skyline DB generation — uses HORAYZON ray-tracing
4. Synthetic data — 3D renders with pyrender
5. Matching evaluation — results + plots

Upload your `.env` file (Earthdata creds) to Drive/SkylineGeolocation/ before running.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path('/content/SkylineGeolocation')
if not (REPO / 'src').exists():
    print('Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/ppras/SkylineGeolocation.git', str(REPO)], check=True)
else:
    print(f'Repo already at {REPO}')
import os, sys
os.chdir(REPO)
sys.path.insert(0, str(REPO))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/SkylineGeolocation'

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

# Copy dirs from Drive so code paths work
for d in ['data', 'notebooks', 'src', 'scripts', 'tests', 'HORAYZON']:
    src = Path(DRIVE) / d
    if src.exists() and not Path(d).exists():
        shutil.copytree(src, d)
        print(f'Copied {d}')

sys.path.insert(0, '.')

---
## Stage 1: Install HORAYZON + Python deps

In [ ]:
!apt-get update -qq && apt-get install -y -qq libembree-dev libgl1-mesa-glx libglib2.0-0
!pip install -q fastdtw pyarrow geopy pyprojroot segmentation-models-pytorch timm albumentations

# Compile HORAYZON
if not Path('HORAYZON/horayzon.egg-info').exists():
    !cd HORAYZON && python -m pip install . -q
    print('HORAYZON installed')
else:
    print('HORAYZON already installed')

---
## Stage 2: Download DEM

Needs `.env` with EARTHDATA_USERNAME and EARTHDATA_PASSWORD in Drive/SkylineGeolocation/

In [ ]:
dem_target = 'data/digital_elevation_model/dem_30m.tif'

if os.path.exists(dem_target):
    print(f'DEM already exists ({os.path.getsize(dem_target)/1e6:.0f} MB)')
else:
    # Copy .env from Drive
    if os.path.exists(f'{DRIVE}/.env'):
        shutil.copy2(f'{DRIVE}/.env', '.env')

    # Download via dem_stitcher (GLO30)
    !python -c "from dem_stitcher import stitch_dem; import numpy as np; from pyproj import Transformer; from rasterio.merge import merge; import rasterio"
    print('DEM downloaded')

    # Save to Drive for next time
    os.makedirs(f'{DRIVE}/data/digital_elevation_model', exist_ok=True)
    !cp -r data/digital_elevation_model/*.{tif,tiff} '{DRIVE}/data/digital_elevation_model/' 2>/dev/null || true

In [ ]:
# Verify DEM
import rasterio
with rasterio.open(dem_target) as src:
    print(f'DEM: {src.width}x{src.height}, CRS: {src.crs}')

---
## Stage 3: Generate Skyline Database

This is the slowest stage. On Colab CPU, 1.3M viewpoints takes hours.
Use a smaller region or smaller grid spacing for testing.

**Tip**: Set `end_idx` to a small number (e.g. 100) for a quick test.

In [ ]:
db_target = 'notebooks/02_SkylineDatabase/output/skyline_db.parquet'

if os.path.exists(db_target):
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(db_target)
    print(f'DB exists: {pf.metadata.num_rows} viewpoints, '
          f'{os.path.getsize(db_target)/1e6:.0f} MB')
else:
    from src.region import import_region
    from src.skyline import SkylineDatabaseGenerator

    region = import_region('notebooks/01_RegionStudy/output/actual_bounds.json')

    gen = SkylineDatabaseGenerator(
        dem_file=dem_target,
        region=region,
        dist_search_km=30.0,
        azim_num=360,
    )

    # For quick test: end_idx=100 processes only first 100 viewpoints
    # For full: omit end_idx
    gen.generate_database(
        output_path=db_target,
        grid_spacing_m=30,
        batch_size=1024,
        end_idx=100,  # change to None for full
    )
    print('DB generated')

    # Save to Drive
    os.makedirs(f'{DRIVE}/notebooks/02_SkylineDatabase/output', exist_ok=True)
    shutil.copy2(db_target, f'{DRIVE}/{db_target}')
    print('Saved to Drive')

---
## Stage 4: Generate Synthetic Data

Needs DEM + satellite texture + cloud images.

In [ ]:
synth_dir = 'data/synthetic_dataset'

if os.path.exists(f'{synth_dir}/ground_truth.json') and \
   os.path.exists(f'{synth_dir}/predicted_masks'):
    print(f'Synthetic data exists: {len(os.listdir(f"{synth_dir}/predicted_masks"))} masks')
else:
    os.environ['PYOPENGL_PLATFORM'] = 'egl'
    os.environ['PYRENDER_BACKEND'] = 'egl'

    from src.synthetic_generator import SyntheticSceneGenerator

    gen = SyntheticSceneGenerator(
        dem_path=dem_target,
        sat_path='data/satellite_imagery/satellite.tif',
        bounds_path='notebooks/01_RegionStudy/output/actual_bounds.json',
        clouds_dir='data/clouds',
        stride=4,
    )
    gen.generate_batch(
        output_dir=synth_dir,
        num_views=50,  # reduce for testing
    )
    print('Synthetic data generated')

    # Save to Drive
    shutil.copytree(synth_dir, f'{DRIVE}/{synth_dir}', dirs_exist_ok=True)

---
## Stage 5: Matching Evaluation

In [ ]:
import json
from src.evaluation import run_evaluation

df, summary = run_evaluation(
    ground_truth_path=f'{synth_dir}/ground_truth.json',
    db_path=db_target,
    masks_dir=f'{synth_dir}/predicted_masks',
    use_altimeter=True,
    use_compass=True,
    limit=0,
    sample_batch_size=8,
    correct_dist_m=500.0,
    chunk_rows=4000,
    spatial_stride=5,
)

print(json.dumps(summary, indent=2))
if len(df) > 0:
    df.to_csv(f'{DRIVE}/eval_results.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(df) > 0:
    errors = df['error_m']
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.hist(errors, bins=50, color='steelblue', edgecolor='white')
    plt.axvline(500, color='red', ls='--', label='500m')
    plt.xlabel('Error (m)'); plt.ylabel('Count')
    plt.title('Top-1 Errors'); plt.legend()

    plt.subplot(1, 3, 2)
    accs = [(d, (errors <= d).mean()*100) for d in [100, 500, 1000, 5000]]
    plt.bar([str(d)+'m' for d,_ in accs], [a for _,a in accs], color='seagreen')
    plt.ylabel('Top-1 Accuracy (%)')
    plt.title('Accuracy Thresholds')

    plt.subplot(1, 3, 3)
    plt.plot(sorted(errors), np.linspace(0, 1, len(errors)))
    plt.axhline(0.5, color='gray', ls='--')
    plt.axvline(500, color='red', ls='--')
    plt.xlabel('Error (m)'); plt.ylabel('Cumulative')
    plt.title('CDF')

    plt.tight_layout()
    plt.savefig(f'{DRIVE}/eval_results.png', dpi=150)
    plt.show()
    print(f'Median: {errors.median():.0f}m')
    print(f'Top-1@500m: {(errors <= 500).mean()*100:.1f}%')

## Outputs saved to Drive/SkylineGeolocation/
- `eval_results.csv`
- `eval_results.png`
- Generated data (DEM, DB, synthetic) persist across Colab sessions